# Month 3 - Herramientas ETL y Proyecto Final

## Week 1 - ETL con Python y SQL

### Extracción de Datos con Python

**Conexiones a Bases de Datos**

¿Cómo conectar Python con bases de datos?

Python puede conectarse a cualquier base de datos usando librerías específicas o ORMs como SQLAlchemy.

Tipos de conexiones:

- PostgreSQL: psycopg2 o SQLAlchemy
- MySQL: pymysql o SQLAlchemy
- SQLite: Librería estándar de Python
- SQL Server: pyodbc

Ejemplo básico de conexión:


```
import sqlite3

# Conexión simple a SQLite
conn = sqlite3.connect('mi_base.db')
cursor = conn.cursor()

# Ejecutar consulta
cursor.execute("SELECT * FROM usuarios")
resultados = cursor.fetchall()

conn.close()
```


**Extracción desde APIs**

Cómo obtener datos de APIs web?

Las APIs REST son una fuente común de datos. Python usa la librería requests para hacer llamadas HTTP.

Conceptos importantes:

- Métodos HTTP: GET (obtener), POST (enviar), PUT (actualizar)
- Autenticación: API keys, tokens, OAuth
- Rate limiting: Límites de llamadas por tiempo
- Paginación: Múltiples páginas de resultados

Ejemplo de llamada API:


```
import requests

# Llamada simple a API
response = requests.get('https://api.ejemplo.com/datos')
if response.status_code == 200:
    datos = response.json()
    print(f"Obtenidos {len(datos)} registros")
```



**Lectura de Archivos**

¿Cómo leer diferentes formatos de archivos?

Python puede leer archivos de datos en múltiples formatos usando librerías especializadas.

Formatos comunes:

- CSV: Archivos de texto separados por comas
- JSON: Datos estructurados en formato JavaScript
- Excel: Hojas de cálculo (.xlsx, .xls)
- Parquet: Formato columnar eficiente

Ejemplo de lectura de CSV:


```
import csv

# Leer archivo CSV
with open('datos.csv', 'r') as archivo:
    lector = csv.DictReader(archivo)
    for fila in lector:
        print(f"Nombre: {fila['nombre']}, Edad: {fila['edad']}")
```



#### Ejemplo: Extraer datos de múltiples fuentes

Leer datos de un archivo CSV:

In [ ]:
import csv

def leer_csv(ruta_archivo):
    datos = []
    with open(ruta_archivo, 'r') as f:
        lector = csv.DictReader(f)
        for fila in lector:
            datos.append(fila)
    return datos

# Uso
clientes = leer_csv('clientes.csv')
print(f"Leídos {len(clientes)} clientes")

Simular extracción de API:

In [ ]:
import json

def extraer_api_simulada():
    # Simular respuesta de API
    datos_api = {
        "productos": [
            {"id": 1, "nombre": "Producto A", "precio": 100},
            {"id": 2, "nombre": "Producto B", "precio": 200}
        ]
    }
    return datos_api["productos"]

productos = extraer_api_simulada()
print(f"Extraídos {len(productos)} productos")

Extraídos 2 productos


Conectar a base de datos SQLite:

In [ ]:
import sqlite3

def conectar_base_datos():
    conn = sqlite3.connect(':memory:')  # Base temporal
    cursor = conn.cursor()

    # Crear tabla
    cursor.execute('''
        CREATE TABLE ventas (
            id INTEGER PRIMARY KEY,
            producto TEXT,
            cantidad INTEGER
        )
    ''')

    # Insertar datos de ejemplo
    cursor.execute("INSERT INTO ventas VALUES (1, 'Producto A', 10)")
    cursor.execute("INSERT INTO ventas VALUES (2, 'Producto B', 5)")

    # Leer datos
    cursor.execute("SELECT * FROM ventas")
    resultados = cursor.fetchall()

    conn.close()
    return resultados

ventas = conectar_base_datos()
print(f"Encontradas {len(ventas)} ventas")

Encontradas 2 ventas


### Transformaciones Básicas con Pandas

**Limpieza de Datos**

¿Qué significa limpiar datos?

La limpieza de datos corrige errores, inconsistencias y problemas que impiden el análisis efectivo.

Problemas comunes:

- Valores faltantes: Datos no disponibles
- Duplicados: Registros repetidos
- Formatos inconsistentes: Fechas, números en diferentes formatos
- Outliers: Valores extremos que pueden ser errores

Ejemplo de limpieza básica:


```
import pandas as pd

# Crear DataFrame con datos sucios
datos = pd.DataFrame({
    'nombre': ['Ana', 'Juan', None, 'María'],
    'edad': [25, None, 30, 25],
    'ciudad': ['Madrid', 'Barcelona', 'Madrid', 'madrid']
})

# Limpiar datos
datos_limpios = datos.copy()
datos_limpios = datos_limpios.dropna()  # Eliminar filas con NaN
datos_limpios['ciudad'] = datos_limpios['ciudad'].str.lower()  # Normalizar texto
```



**Manejo de Valores Faltantes**

¿Cómo manejar datos faltantes?

Los valores faltantes (NaN, None, null) son inevitables. Hay varias estrategias para manejarlos.

Estrategias principales:

- Eliminar: Quitar filas/columnas con muchos faltantes
- Imputar: Rellenar con valores calculados (media, mediana, moda)
- Flag: Crear columna indicando si faltaba el dato original
- Preservar: Mantener como está si es informativo

Ejemplo de imputación:


```
# Imputar valores faltantes
datos['edad'] = datos['edad'].fillna(datos['edad'].mean())  # Media
datos['ciudad'] = datos['ciudad'].fillna('desconocido')  # Valor fijo
```



**Normalización y Validación**

¿Por qué normalizar datos?

La normalización asegura que los datos tengan formatos consistentes y valores válidos.

Técnicas de normalización:

- Texto: Convertir a minúsculas, quitar espacios extra
- Fechas: Formato consistente (YYYY-MM-DD)
- Números: Escala común, quitar separadores de miles
- Categorías: Valores estandarizados


1. Tipos de Validación

No todos los datos se validan igual. Depende de qué representan:

- **Validación de Tipo**: Asegurarse de que un número sea un número y no un texto (ej. que la columna precio no contenga la palabra "gratis").

- **Validación de Rango**: Comprobar que los valores tengan sentido lógico (ej. una edad entre 0 y 120, o un mes entre 1 y 12).

- **Validación de Formato**: Muy común en correos electrónicos, números de teléfono o códigos postales mediante el uso de Regex (Expresiones Regulares).

- **Validación de Consistencia**: Que un dato guarde relación con otro (ej. que la fecha_de_entrega no sea anterior a la fecha_de_compra).

Ejemplo de validación:


```
def validar_datos(df):
    errores = []
    
    # Validar edades razonables
    if (df['edad'] < 0).any() or (df['edad'] > 120).any():
        errores.append("Edades fuera de rango")
    
    # Validar emails
    import re
    patron_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    if not df['email'].str.match(patron_email).all():
        errores.append("Emails inválidos")
    
    return errores
```



#### Ejemplo: Limpiar y validar un dataset de ventas

Dataset con problemas:

In [ ]:
import pandas as pd
import numpy as np

# Crear datos de ejemplo con problemas
ventas = pd.DataFrame({
    'producto': ['A', 'B', None, 'A', 'C'],
    'precio': [100, None, 150, 100, 200],
    'cantidad': [1, 2, None, 1, 3],
    'fecha': ['2024-01-01', None, '2024-01-03', '2024-01-01', 'invalid']
})

print("Datos originales:")
print(ventas)
print(f"Valores faltantes por columna:\n{ventas.isnull().sum()}")

Datos originales:
  producto  precio  cantidad       fecha
0        A   100.0       1.0  2024-01-01
1        B     NaN       2.0        None
2     None   150.0       NaN  2024-01-03
3        A   100.0       1.0  2024-01-01
4        C   200.0       3.0     invalid
Valores faltantes por columna:
producto    1
precio      1
cantidad    1
fecha       1
dtype: int64


Limpiar datos:

In [ ]:
def limpiar_datos_ventas(df):
    df_limpio = df.copy()

    # 1. Eliminar duplicados
    df_limpio = df_limpio.drop_duplicates()

    # 2. Imputar valores faltantes
    df_limpio['precio'] = df_limpio['precio'].fillna(df_limpio['precio'].median())
    df_limpio['cantidad'] = df_limpio['cantidad'].fillna(1)  # Asumir cantidad mínima

    # 3. Eliminar filas con producto faltante
    df_limpio = df_limpio.dropna(subset=['producto'])

    # 4. Corregir fechas inválidas
    df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'], errors='coerce')
    df_limpio = df_limpio.dropna(subset=['fecha'])

    # 5. Calcular total
    df_limpio['total'] = df_limpio['precio'] * df_limpio['cantidad']

    return df_limpio

ventas_limpias = limpiar_datos_ventas(ventas)
print("\nDatos limpios:")
print(ventas_limpias)
print(f"\nRegistros finales: {len(ventas_limpias)}")


Datos limpios:
  producto  precio  cantidad      fecha  total
0        A   100.0       1.0 2024-01-01  100.0

Registros finales: 1


Validar datos limpios:

In [ ]:
def validar_ventas_limpias(df):
    validaciones = {
        'sin_faltantes': df.isnull().sum().sum() == 0,
        'precios_positivos': (df['precio'] > 0).all(),
        'cantidades_positivas': (df['cantidad'] > 0).all(),
        'fechas_validas': pd.api.types.is_datetime64_any_dtype(df['fecha']),
        'total_correcto': np.allclose(df['total'], df['precio'] * df['cantidad'])
    }

    print("Validaciones:")
    for check, passed in validaciones.items():
        status = "✅" if passed else "❌"
        print(f"  {status} {check}")

    return all(validaciones.values())

es_valido = validar_ventas_limpias(ventas_limpias)
print(f"\nDataset válido: {es_valido}")

Validaciones:
  ✅ sin_faltantes
  ✅ precios_positivos
  ✅ cantidades_positivas
  ✅ fechas_validas
  ✅ total_correcto

Dataset válido: True


### Transformaciones Avanzadas y Enriquecimiento

**Joins y Merge**

¿Cómo combinar datos de múltiples fuentes?

Los joins permiten combinar datos relacionados de diferentes tablas o datasets.

Tipos de joins:

- Inner join: Solo filas que existen en ambas tablas
- Left join: Todas las filas de la tabla izquierda + matches de la derecha
- Right join: Todas las filas de la tabla derecha + matches de la izquierda
- Outer join: Todas las filas de ambas tablas

Ejemplo de join:


```
import pandas as pd

# Dataset de clientes
clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4],
    'nombre': ['Ana', 'Juan', 'María', 'Pedro']
})

# Dataset de pedidos
pedidos = pd.DataFrame({
    'cliente_id': [1, 1, 3, 5],
    'producto': ['A', 'B', 'C', 'D'],
    'cantidad': [2, 1, 3, 1]
})

# Join para combinar información
pedidos_clientes = pd.merge(
    pedidos,
    clientes,
    on='cliente_id',
    how='left'
)

print(pedidos_clientes)
```



**Agregaciones y Cálculos Derivados**

¿Cómo crear métricas calculadas?

Las agregaciones resumen datos y los cálculos derivados crean nuevas métricas basadas en datos existentes.

Operaciones comunes:

- Sumar: Totales por categoría
- Contar: Número de elementos
- Promediar: Valores medios
- Agrupar: Análisis por segmentos

Ejemplo de agregación:


```
# Agregaciones por cliente
resumen_cliente = pedidos_clientes.groupby('cliente_id').agg({
    'cantidad': 'sum',
    'producto': 'count'
}).rename(columns={
    'cantidad': 'total_cantidad',
    'producto': 'numero_pedidos'
})

print(resumen_cliente)
```



**Validaciones de Integridad**

¿Cómo asegurar consistencia de datos?

Las validaciones de integridad verifican que los datos sean coherentes y cumplan reglas de negocio.

Tipos de validación:

- Referencial: Claves foráneas existen
- Dominio: Valores dentro de rangos válidos
- Consistencia: Datos coherentes entre campos
- Negocio: Reglas específicas del dominio

Ejemplo de validación:


```
def validar_integridad(df):
    errores = []
    
    # Validar que clientes referenciados existen
    clientes_validos = set(clientes['cliente_id'])
    pedidos_invalidos = df[~df['cliente_id'].isin(clientes_validos)]
    
    if len(pedidos_invalidos) > 0:
        errores.append(f"Pedidos con clientes inexistentes: {len(pedidos_invalidos)}")
    
    # Validar cantidades positivas
    if (df['cantidad'] <= 0).any():
        errores.append("Cantidades no positivas encontradas")
    
    return errores

errores = validar_integridad(pedidos_clientes)
print("Errores de integridad:", errores)
```



#### Ejemplo: Transformaciones avanzadas en dataset de e-commerce

Datos base:

In [ ]:
import pandas as pd
import numpy as np

# Clientes
clientes = pd.DataFrame({
    'cliente_id': range(1, 6),
    'nombre': ['Ana', 'Juan', 'María', 'Pedro', 'Laura'],
    'segmento': ['Premium', 'Regular', 'Premium', 'Regular', 'VIP']
})

# Pedidos
pedidos = pd.DataFrame({
    'pedido_id': range(1, 11),
    'cliente_id': np.random.choice(range(1, 6), 10),
    'producto': np.random.choice(['A', 'B', 'C', 'D'], 10),
    'precio': np.random.uniform(50, 500, 10).round(2),
    'fecha': pd.date_range('2024-01-01', periods=10)
})

print("Clientes y pedidos cargados")

Clientes y pedidos cargados


Enriquecer datos con joins:

In [ ]:
# Unir pedidos con información de clientes
pedidos_enriquecidos = pd.merge(
    pedidos,
    clientes,
    on='cliente_id',
    how='left'
)

print("Pedidos con información de clientes:")
print(pedidos_enriquecidos.head())

Pedidos con información de clientes:
   pedido_id  cliente_id producto  precio      fecha nombre segmento
0          1           4        B  312.98 2024-01-01  Pedro  Regular
1          2           1        A   51.79 2024-01-02    Ana  Premium
2          3           4        A   71.05 2024-01-03  Pedro  Regular
3          4           1        A  483.18 2024-01-04    Ana  Premium
4          5           2        D  338.69 2024-01-05   Juan  Regular


Calcular métricas derivadas:

In [ ]:
# Calcular métricas por cliente
metricas_cliente = pedidos_enriquecidos.groupby(['cliente_id', 'nombre', 'segmento']).agg({
    'pedido_id': 'count',
    'precio': ['sum', 'mean', 'max'],
    'fecha': 'max'  # Última compra
}).round(2)

# Aplanar columnas multi-nivel
metricas_cliente.columns = ['num_pedidos', 'total_gastado', 'gasto_promedio', 'gasto_maximo', 'ultima_compra']
metricas_cliente = metricas_cliente.reset_index()

print("\nMétricas por cliente:")
print(metricas_cliente)


Métricas por cliente:
   cliente_id nombre segmento  num_pedidos  total_gastado  gasto_promedio  \
0           1    Ana  Premium            4         870.85          217.71   
1           2   Juan  Regular            2         685.38          342.69   
2           4  Pedro  Regular            2         384.03          192.02   
3           5  Laura      VIP            2         660.81          330.40   

   gasto_maximo ultima_compra  
0        483.18    2024-01-10  
1        346.69    2024-01-06  
2        312.98    2024-01-03  
3        420.47    2024-01-09  


Validar reglas de negocio:

In [ ]:
def validar_reglas_negocio(df):
    validaciones = []

    # VIP deben tener al menos 2 pedidos
    vip_insuficientes = df[(df['segmento'] == 'VIP') & (df['num_pedidos'] < 2)]
    if len(vip_insuficientes) > 0:
        validaciones.append(f"VIPs con pocos pedidos: {len(vip_insuficientes)}")

    # Premium no deben exceder gasto máximo
    premium_excesivos = df[(df['segmento'] == 'Premium') & (df['gasto_maximo'] > 800)]
    if len(premium_excesivos) > 0:
        validaciones.append(f"Premiums con gastos excesivos: {len(premium_excesivos)}")

    return validaciones

reglas_incumplidas = validar_reglas_negocio(metricas_cliente)
print(f"\nReglas de negocio incumplidas: {reglas_incumplidas}")


Reglas de negocio incumplidas: []


### Carga de Datos y Estrategias de Destino

**Estrategias de Carga**

¿Cómo cargar datos eficientemente?

Existen diferentes estrategias dependiendo del volumen de datos y frecuencia de actualización.

Tipos de carga:

- Completa (Full Load): Reemplazar todos los datos
- Incremental: Solo actualizar cambios
- Upsert: Insertar nuevos, actualizar existentes
- Streaming: Carga continua en tiempo real

Cuándo usar cada estrategia:

- Full Load: Datasets pequeños, cambios masivos
- Incremental: Grandes volúmenes, cambios frecuentes
- Upsert: Datos con claves naturales
- Streaming: Requisitos de baja latencia


**Carga a Bases de Datos SQL**

¿Cómo escribir datos a bases de datos?

La carga a SQL requiere considerar constraints, índices y performance.

Consideraciones importantes:

- Transacciones: Agrupar operaciones para consistencia
- Batch size: Tamaño óptimo de lotes
- Error handling: Qué hacer con registros problemáticos
- Índices: Desactivar durante carga masiva

Ejemplo de carga SQL:


```
import pandas as pd
from sqlalchemy import create_engine

def cargar_a_postgresql(df, tabla, engine):
    try:
        # Cargar datos
        df.to_sql(
            tabla,
            engine,
            if_exists='append',  # 'replace' para full load
            index=False,
            chunksize=1000  # Batch size
        )
        print(f"Cargados {len(df)} registros a {tabla}")
        return True
    except Exception as e:
        print(f"Error en carga: {e}")
        return False
```



**Carga a Formatos Analíticos**

¿Cómo optimizar para análisis?

Los formatos analíticos priorizan velocidad de lectura sobre escritura.

Formatos eficientes:

- Parquet: Columnar, comprimido, rápido para analytics
- Delta Lake: Transaccional, con historial de cambios
- Iceberg: Formato abierto con evolución de esquemas

Ejemplo de carga a Parquet:


```
def cargar_a_parquet(df, ruta_archivo):
    try:
        df.to_parquet(
            ruta_archivo,
            engine='pyarrow',
            compression='snappy',  # Buena compresión + velocidad
            index=False
        )
        print(f"Datos guardados en {ruta_archivo}")
        return True
    except Exception as e:
        print(f"Error guardando Parquet: {e}")
        return False
```



#### Ejemplo: Implementar diferentes estrategias de carga

Preparar datos de ejemplo:

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Generar datos de ventas
np.random.seed(42)
ventas = pd.DataFrame({
    'venta_id': range(1, 1001),
    'cliente_id': np.random.randint(1, 101, 1000),
    'producto_id': np.random.randint(1, 51, 1000),
    'cantidad': np.random.randint(1, 11, 1000),
    'precio_unitario': np.round(np.random.uniform(10, 500, 1000), 2),
    'fecha_venta': pd.date_range('2024-01-01', periods=1000, freq='1H'),
    'updated_at': datetime.now()
})

ventas['total'] = ventas['cantidad'] * ventas['precio_unitario']

print(f"Generados {len(ventas)} registros de ventas")
print(ventas.head())

Generados 1000 registros de ventas
   venta_id  cliente_id  producto_id  cantidad  precio_unitario  \
0         1          52           34         4           405.14   
1         2          93           47        10           235.03   
2         3          15            8         6            35.46   
3         4          72           40         7           395.28   
4         5          61           49         2           108.67   

          fecha_venta                 updated_at    total  
0 2024-01-01 00:00:00 2026-01-02 21:37:25.059445  1620.56  
1 2024-01-01 01:00:00 2026-01-02 21:37:25.059445  2350.30  
2 2024-01-01 02:00:00 2026-01-02 21:37:25.059445   212.76  
3 2024-01-01 03:00:00 2026-01-02 21:37:25.059445  2766.96  
4 2024-01-01 04:00:00 2026-01-02 21:37:25.059445   217.34  


/tmp/ipython-input-1391396460.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'fecha_venta': pd.date_range('2024-01-01', periods=1000, freq='1H'),


Carga completa (full load):

In [ ]:
import sqlite3

def carga_completa_sqlite(df, tabla):
    conn = sqlite3.connect(':memory:')

    # Crear tabla
    conn.execute(f'''
        CREATE TABLE {tabla} (
            venta_id INTEGER PRIMARY KEY,
            cliente_id INTEGER,
            producto_id INTEGER,
            cantidad INTEGER,
            precio_unitario REAL,
            total REAL,
            fecha_venta TEXT,
            updated_at TEXT
        )
    ''')

    # Insertar datos
    df.to_sql(tabla, conn, if_exists='replace', index=False)

    # Verificar
    cursor = conn.execute(f"SELECT COUNT(*) FROM {tabla}")
    count = cursor.fetchone()[0]

    conn.close()
    return count

registros_cargados = carga_completa_sqlite(ventas, 'ventas_completas')
print(f"Carga completa: {registros_cargados} registros")

Carga completa: 1000 registros


Carga incremental (simulada):

In [ ]:
def carga_incremental(df, archivo_parquet, ultimo_id=0):
    # Simular carga incremental: solo registros nuevos
    nuevos_registros = df[df['venta_id'] > ultimo_id]

    if len(nuevos_registros) > 0:
        try:
            # En producción, leer archivo existente y append
            nuevos_registros.to_parquet(
                archivo_parquet,
                engine='pyarrow',
                index=False
            )
            print(f"Carga incremental: {len(nuevos_registros)} nuevos registros")
            return len(nuevos_registros)
        except Exception as e:
            print(f"Error en carga incremental: {e}")
            return 0
    else:
        print("No hay nuevos registros para cargar")
        return 0

nuevos_cargados = carga_incremental(ventas, 'ventas_incremental.parquet', ultimo_id=500)
print(f"Registros nuevos agregados: {nuevos_cargados}")

Carga incremental: 500 nuevos registros
Registros nuevos agregados: 500


Comparar estrategias:

In [ ]:
import time

def comparar_estrategias_carga():
    estrategias = {}

    # Medir carga completa
    start = time.time()
    carga_completa_sqlite(ventas, 'ventas_test')
    estrategias['completa'] = time.time() - start

    # Medir carga incremental (simulada)
    start = time.time()
    carga_incremental(ventas, 'ventas_inc_test.parquet', ultimo_id=800)
    estrategias['incremental'] = time.time() - start

    print("Comparación de estrategias:")
    print(".2f")
    print(".2f")

    return estrategias

resultados = comparar_estrategias_carga()

Carga incremental: 200 nuevos registros
Comparación de estrategias:
.2f
.2f


### Manejo de Errores y Logging en ETL

**Estrategias de Manejo de Errores**

¿Por qué fallan los pipelines ETL?

Los pipelines pueden fallar por múltiples razones: datos corruptos, conexiones caídas, recursos insuficientes, lógica errónea.

Tipos de errores comunes:

- Datos: Valores faltantes, formatos inválidos, duplicados
- Conectividad: APIs caídas, bases de datos inaccesibles
- Recursos: Memoria insuficiente, timeouts
- Lógica: Errores en transformaciones, cálculos incorrectos

Estrategias de manejo:

```
def procesar_con_errores(operacion, max_reintentos=3):
    """Template para manejar errores con reintentos"""
    for intento in range(max_reintentos):
        try:
            return operacion()
        except Exception as e:
            if intento == max_reintentos - 1:
                print(f"Error definitivo: {e}")
                raise e
            else:
                print(f"Intento {intento + 1} falló: {e}. Reintentando...")
                time.sleep(2 ** intento)  # Exponential backoff
```



**Logging Efectivo**

¿Cómo hacer logging útil?

El logging debe ayudar a debuggear problemas y monitorear el estado del pipeline.

Niveles de logging:

- DEBUG: Información detallada para desarrollo
- INFO: Eventos normales del pipeline
- WARNING: Situaciones que requieren atención
- ERROR: Errores que impiden funcionamiento
- CRITICAL: Errores que requieren intervención inmediata

Ejemplo de logging configurado:


```
import logging

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_pipeline.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('etl_pipeline')

# Uso en pipeline
def etapa_etl(datos):
    logger.info(f"Iniciando procesamiento de {len(datos)} registros")
    
    try:
        # Procesamiento
        resultado = procesar_datos(datos)
        logger.info(f"Procesamiento completado: {len(resultado)} registros")
        return resultado
        
    except Exception as e:
        logger.error(f"Error en procesamiento: {e}")
        raise e
```

**¿Qué incluir en los logs para un Debugging efectivo?**

Un log no es solo una lista de eventos; es el mapa que te permite reconstruir el pasado. Para que sea realmente útil, cada entrada de log debería responder a estas preguntas:

1. ¿Cuándo? (Timestamp): Fecha y hora exacta (preferiblemente en UTC) con milisegundos.

2. ¿Quién? (Logger/Module): Qué parte del código está hablando (ej: extractor_ventas vs validador_precios).

3. ¿Qué tan grave? (Level): INFO, WARNING, ERROR o CRITICAL.

4. Contexto de los Datos: No digas solo "Error al procesar". Di: "Error al procesar el pedido_id: 504 del cliente Ana".

5. El Rastro del Error (Traceback): Si ocurre una excepción, captura el tipo de error y la línea exacta donde falló.

6. Métricas de Rendimiento: Tiempo que tardó la operación (ej: "Extracción completada en 4.2 segundos").

7. Tip Pro: Usa Logging Estructurado (formato JSON) si vas a enviar tus logs a herramientas externas como Datadog o ELK Stack. Esto permite filtrar errores por ID de cliente o por tipo de error en segundos.

**Validaciones Post-ETL**

¿Cómo verificar que el ETL funcionó correctamente?

Las validaciones post-ETL aseguran que los datos cargados sean correctos y completos.

Validaciones importantes:

- Conteo de registros: ¿Llegaron todos los datos?
- Suma de controles: ¿Los totales coinciden?
- Calidad de datos: ¿Se mantuvieron las reglas?
    - Integridad referencial: ¿Las relaciones son válidas?

Ejemplo de validaciones:


```
def validar_carga(origen_df, destino_tabla, engine):
    """Validar que la carga fue exitosa"""
    validaciones = {}
    
    # Contar registros
    origen_count = len(origen_df)
    destino_count = pd.read_sql(f"SELECT COUNT(*) FROM {destino_tabla}", engine).iloc[0, 0]
    validaciones['conteo_registros'] = origen_count == destino_count
    
    # Suma de totales
    origen_total = origen_df['total'].sum()
    destino_total = pd.read_sql(f"SELECT SUM(total) FROM {destino_tabla}", engine).iloc[0, 0]
    validaciones['suma_totales'] = abs(origen_total - destino_total) < 0.01
    
    # Valores faltantes
    faltantes = pd.read_sql(f"SELECT COUNT(*) FROM {destino_tabla} WHERE total IS NULL", engine).iloc[0, 0]
    validaciones['sin_faltantes'] = faltantes == 0
    
    return validaciones
```



#### Ejemplo: Construir pipeline ETL con manejo de errores completo

Configurar logging:

In [ ]:
import logging
import time
from functools import wraps

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_ecommerce.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('etl_ecommerce')

def log_etapa(etapa):
    """Decorator para logging de etapas"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            logger.info(f"🚀 Iniciando {etapa}")
            start_time = time.time()

            try:
                result = func(*args, **kwargs)
                duration = time.time() - start_time
                logger.info(f"✅ {etapa} completada en {duration:.2f}s")
                return result
            except Exception as e:
                duration = time.time() - start_time
                logger.error(f"💥 {etapa} falló en {duration:.2f}s: {e}")
                raise e

        return wrapper
    return decorator

Pipeline ETL con error handling:

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Any

class ETLPipeline:
    def __init__(self):
        self.logger = logger
        self.errores = []

    @log_etapa("extracción de datos")
    def extract(self) -> pd.DataFrame:
        """Extraer datos con manejo de errores"""
        try:
            # Simular extracción (podría fallar)
            if np.random.random() < 0.1:  # 10% chance de error
                raise ConnectionError("Error de conexión a fuente de datos")

            # Datos de ejemplo
            datos = pd.DataFrame({
                'orden_id': range(1, 101),
                'cliente_id': np.random.randint(1, 21, 100),
                'producto': np.random.choice(['A', 'B', 'C', 'D'], 100),
                'cantidad': np.random.randint(1, 6, 100),
                'precio': np.round(np.random.uniform(10, 200, 100), 2)
            })

            self.logger.info(f"Extraídos {len(datos)} registros")
            return datos

        except Exception as e:
            self.errores.append(f"Extract: {e}")
            raise e

    @log_etapa("transformación de datos")
    def transform(self, datos: pd.DataFrame) -> pd.DataFrame:
        """Transformar datos con validaciones"""
        try:
            df = datos.copy()

            # Validar datos de entrada
            if df.empty:
                raise ValueError("No hay datos para transformar")

            # Transformaciones
            df['total'] = df['cantidad'] * df['precio']
            df['categoria_precio'] = pd.cut(
                df['precio'],
                bins=[0, 50, 100, 200],
                labels=['Bajo', 'Medio', 'Alto']
            )

            # Validar transformaciones
            if df['total'].isnull().any():
                raise ValueError("Transformación produjo valores nulos")

            self.logger.info(f"Transformados {len(df)} registros")
            return df

        except Exception as e:
            self.errores.append(f"Transform: {e}")
            raise e

    @log_etapa("carga de datos")
    def load(self, datos: pd.DataFrame) -> bool:
        """Cargar datos con verificación"""
        try:
            # Simular carga (podría fallar)
            if np.random.random() < 0.05:  # 5% chance de error
                raise Exception("Error de conexión a base de datos")

            # En producción: datos.to_sql('ventas', engine, if_exists='append')
            self.logger.info(f"Cargados {len(datos)} registros exitosamente")

            # Validar carga
            registros_esperados = len(datos)
            registros_cargados = len(datos)  # Simulado

            if registros_cargados != registros_esperados:
                raise ValueError(f"Carga incompleta: {registros_cargados}/{registros_esperados}")

            return True

        except Exception as e:
            self.errores.append(f"Load: {e}")
            raise e

    def ejecutar_pipeline(self) -> Dict[str, Any]:
        """Ejecutar pipeline completo con manejo de errores"""
        self.logger.info("🎯 Iniciando pipeline ETL completo")

        try:
            # Extract
            datos_crudo = self.extract()

            # Transform
            datos_transformados = self.transform(datos_crudo)

            # Load
            exito = self.load(datos_transformados)

            resultado = {
                'exito': True,
                'registros_procesados': len(datos_transformados),
                'errores': self.errores
            }

            self.logger.info("🎉 Pipeline ETL completado exitosamente")
            return resultado

        except Exception as e:
            self.logger.error(f"🚨 Pipeline ETL falló: {e}")

            return {
                'exito': False,
                'error_principal': str(e),
                'errores': self.errores
            }

Ejecutar y validar pipeline:

In [ ]:
# Ejecutar pipeline con diferentes escenarios
pipeline = ETLPipeline()

# Ejecución exitosa
resultado = pipeline.ejecutar_pipeline()

print(f"\nResultado del pipeline:")
print(f"Éxito: {resultado['exito']}")
if resultado['exito']:
    print(f"Registros procesados: {resultado['registros_procesados']}")
else:
    print(f"Error principal: {resultado['error_principal']}")

print(f"Errores registrados: {len(resultado['errores'])}")
for error in resultado['errores']:
    print(f"  - {error}")

# Ejecutar múltiples veces para probar robustez
resultados_multiples = []
for i in range(5):
    print(f"\n--- Ejecución {i+1} ---")
    pipeline_i = ETLPipeline()
    resultado_i = pipeline_i.ejecutar_pipeline()
    resultados_multiples.append(resultado_i['exito'])

exito_rate = sum(resultados_multiples) / len(resultados_multiples)
print(".1%")

ERROR:etl_ecommerce:💥 extracción de datos falló en 0.00s: Error de conexión a fuente de datos
ERROR:etl_ecommerce:🚨 Pipeline ETL falló: Error de conexión a fuente de datos



Resultado del pipeline:
Éxito: True
Registros procesados: 100
Errores registrados: 0

--- Ejecución 1 ---

--- Ejecución 2 ---

--- Ejecución 3 ---

--- Ejecución 4 ---

--- Ejecución 5 ---
.1%


### Ejemplo 2:  Resumen utilizando Parquet

In [ ]:
import pandas as pd

# --- PASO 1: EXTRACCIÓN (Día 1) ---
def extraer_datos():
    # Simulamos la carga de un CSV
    data = {
        'id_pedido': [101, 102, 103, 104],
        'producto': ['Laptop', '  MOUSE', 'Laptop', 'Teclado'],
        'cantidad': [1, 2, None, 1],
        'precio_unitario': [1000, 25, 1000, 50]
    }
    return pd.DataFrame(data)

# --- PASO 2 Y 3: LIMPIEZA Y TRANSFORMACIÓN (Día 2 y 3) ---
def transformar_datos(df):
    # Limpieza: Normalizar texto y llenar nulos
    df['producto'] = df['producto'].str.strip().str.capitalize()
    df['cantidad'] = df['cantidad'].fillna(1) # Asumimos 1 si no hay dato

    # Validación: Solo cantidades positivas
    df = df[df['cantidad'] > 0]

    # Cálculo derivado: Total por línea
    df['total_linea'] = df['cantidad'] * df['precio_unitario']

    # Agregación: Total por producto
    resumen = df.groupby('producto').agg({
        'id_pedido': 'count',
        'total_linea': 'sum'
    }).rename(columns={'id_pedido': 'ventas_realizadas', 'total_linea': 'ingresos_totales'})

    return resumen

# --- PASO 4: CARGA (Día 4) ---
def cargar_datos(df):
    try:
        # Guardamos en Parquet para el equipo de Analytics
        df.to_parquet('reporte_ventas.parquet', index=True)
        print("✅ Proceso ETL finalizado con éxito. Archivo 'reporte_ventas.parquet' creado.")
    except Exception as e:
        print(f"❌ Error en la carga: {e}")

# --- EJECUCIÓN DEL PIPELINE ---
df_inicial = extraer_datos()
df_transformado = transformar_datos(df_inicial)
cargar_datos(df_transformado)

# Ver el resultado final
print("\nVista previa del resultado:")
print(df_transformado)

✅ Proceso ETL finalizado con éxito. Archivo 'reporte_ventas.parquet' creado.

Vista previa del resultado:
          ventas_realizadas  ingresos_totales
producto                                     
Laptop                    2            2000.0
Mouse                     1              50.0
Teclado                   1              50.0


### Ejemplo Completo:

In [ ]:
import pandas as pd
import sqlite3
import logging
import time
from sqlalchemy import create_engine

# --- CONFIGURACIÓN DE LOGGING (Día 5) ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler('pipeline_semana1.log'), logging.StreamHandler()]
)
logger = logging.getLogger('ETL_Master')

# --- 1. EXTRACCIÓN (Día 1) ---
def extraer_datos():
    logger.info("Iniciando fase de EXTRACCIÓN...")
    # Simulamos datos que vienen de un CSV o API
    raw_data = {
        'id_venta': [1, 2, 3, 4, 5],
        'producto': ['Laptop', 'Mouse', '  LAPTOP ', 'Teclado', 'Smartphone'],
        'monto': [1200, 25, 1200, None, 800], # Tenemos un nulo
        'cliente_id': [10, 11, 10, 12, 99]    # El cliente 99 no existe en nuestra base
    }
    return pd.DataFrame(raw_data)

# --- 2. LIMPIEZA Y TRANSFORMACIÓN (Día 2 y 3) ---
def transformar_datos(df):
    logger.info("Iniciando fase de TRANSFORMACIÓN...")

    # Limpieza (Día 2)
    df['producto'] = df['producto'].str.strip().str.capitalize()
    df['monto'] = df['monto'].fillna(df['monto'].median()) # Imputación por mediana

    # Join con tabla de clientes (Día 3)
    clientes_db = pd.DataFrame({
        'cliente_id': [10, 11, 12],
        'nombre_cliente': ['Ana', 'Juan', 'Maria']
    })

    # Left Join para mantener todas las ventas
    df_enriquecido = pd.merge(df, clientes_db, on='cliente_id', how='left')

    # Cálculo derivado (Día 3)
    df_enriquecido['impuesto'] = df_enriquecido['monto'] * 0.15

    return df_enriquecido

# --- 3. CARGA Y VALIDACIÓN (Día 4 y 5) ---
def cargar_y_validar(df):
    logger.info("Iniciando fase de CARGA...")

    # Manejo de Errores y Reintentos (Día 5)
    intentos = 3
    for i in range(intentos):
        try:
            # Conexión a Base de Datos (Día 1/4)
            engine = create_engine('sqlite:///ventas_final.db')

            # Carga Incremental (Día 4)
            df.to_sql('reporte_diario', engine, if_exists='replace', index=False)

            # Validación Post-ETL (Día 5)
            count_db = pd.read_sql("SELECT COUNT(*) FROM reporte_diario", engine).iloc[0,0]
            if count_db == len(df):
                logger.info(f"✅ Validación exitosa: {count_db} registros cargados.")
                return True
            else:
                raise Exception("Discrepancia en el conteo de registros.")

        except Exception as e:
            logger.warning(f"Intento {i+1} falló: {e}")
            time.sleep(1)
            if i == intentos - 1:
                logger.critical("Fallo catastrófico en la carga.")
                return False

# --- EJECUCIÓN DEL PIPELINE COMPLETO ---
if __name__ == "__main__":
    try:
        data_cruda = extraer_datos()
        data_procesada = transformar_datos(data_cruda)

        # Validación de Integridad Referencial (Día 3)
        huerfanos = data_procesada[data_procesada['nombre_cliente'].isna()]
        if not huerfanos.empty:
            logger.warning(f"Se detectaron {len(huerfanos)} ventas de clientes inexistentes.")

        exito = cargar_y_validar(data_procesada)

        if exito:
            logger.info("🚀 Pipeline ejecutado correctamente de punta a punta.")
            print("\n--- VISTA PREVIA DE LOS DATOS PROCESADOS ---")
            print(data_procesada)

    except Exception as e:
        logger.error(f"El pipeline se detuvo por un error inesperado: {e}")


--- VISTA PREVIA DE LOS DATOS PROCESADOS ---
   id_venta    producto   monto  cliente_id nombre_cliente  impuesto
0         1      Laptop  1200.0          10            Ana    180.00
1         2       Mouse    25.0          11           Juan      3.75
2         3      Laptop  1200.0          10            Ana    180.00
3         4     Teclado  1000.0          12          Maria    150.00
4         5  Smartphone   800.0          99            NaN    120.00


## Week 2 - Automatización de Pipelines

### Introducción a Apache Airflow

**¿Qué es Apache Airflow?**

- ¿Por qué necesitamos orquestar pipelines?

    Los pipelines complejos tienen múltiples pasos que deben ejecutarse en orden específico, con dependencias entre ellos.

- Problemas sin orquestación:

    - Tareas manuales propensas a error
    - Dificultad para re-ejecutar pasos específicos
    - Falta de visibilidad del estado del pipeline
    - Problemas de coordinación en equipos

- Solución: Apache Airflow

    - Orquestación: Coordina ejecución de tareas
    - Programación: Ejecuta en horarios específicos
    - Monitoreo: Dashboard para ver estado
    - Reintentos: Automatiza recuperación de fallos


**Arquitectura Básica de Airflow**

1. Componentes principales:

    - Web Server: Interfaz web para monitoreo
    - Scheduler: Programa y ejecuta DAGs
    - Workers: Ejecutan las tareas individuales
    - Metadata Database: Almacena estado y configuración

2. Flujo de trabajo típico:

        Usuario escribe DAG → Scheduler detecta → Worker ejecuta tareas →
        Usuario monitorea en Web UI → Resultados se almacenan

3. Ventajas de la arquitectura:

    - Escalable: Múltiples workers pueden procesar tareas en paralelo
    - Fault-tolerant: Si un worker falla, otros pueden continuar
    - Observable: Web UI muestra estado detallado de todas las tareas


**Conceptos Básicos de DAGs**

¿Qué es un DAG?

DAG = Directed Acyclic Graph (Grafo Dirigido Acíclico)

Características importantes:

- Directed: Las tareas tienen dirección (A → B significa A antes que B)
- Acyclic: No hay ciclos (no puede haber A → B → A)
- Graph: Estructura de nodos conectados

Elementos de un DAG:


```
from airflow import DAG
from airflow.operators.dummy import DummyOperator
from datetime import datetime

# Definición básica de DAG
dag = DAG(
    'mi_primer_dag',
    description='DAG de ejemplo',
    schedule_interval='@daily',
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# Tareas (nodos del grafo)
inicio = DummyOperator(
    task_id='inicio',
    dag=dag
)

fin = DummyOperator(
    task_id='fin',
    dag=dag
)

# Dependencias (flechas del grafo)
inicio >> fin
```



#### Ejemplo: Crear tu primer DAG funcional (Local)

Instalación básica de Airflow:

In [ ]:
# Crear entorno virtual
!python -m venv airflow_env
!source airflow_env/bin/activate

# Instalar Airflow
!pip install apache-airflow

# Inicializar base de datos
!airflow db init

# Crear usuario admin
!airflow users create \
  --username admin \
  --firstname Admin \
  --lastname User \
  --role Admin \
  --email admin@example.com

Crear primer DAG:

In [3]:
# dags/mi_primer_dag.py
from airflow import DAG
from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime, timedelta

def saludar():
    print("¡Hola desde Airflow!")
    return "Saludo completado"

# Definir DAG
dag = DAG(
    'saludo_diario',
    description='DAG que saluda cada día',
    schedule=timedelta(days=1),  # Ejecutar diariamente
    start_date=datetime(2024, 1, 1),
    catchup=False,  # No ejecutar ejecuciones pasadas
    tags=['ejemplo', 'saludo']
)

# Tarea 1: Comando bash
tarea_bash = BashOperator(
    task_id='tarea_bash',
    bash_command='echo "Ejecutando tarea bash a las $(date)"',
    dag=dag
)

# Tarea 2: Función Python
tarea_python = PythonOperator(
    task_id='tarea_python',
    python_callable=saludar,
    dag=dag
)

# Tarea 3: Esperar (simular procesamiento)
tarea_esperar = BashOperator(
    task_id='tarea_esperar',
    bash_command='sleep 5',
    dag=dag
)

# Definir orden de ejecución
tarea_bash >> tarea_python >> tarea_esperar

<Task(BashOperator): tarea_esperar>

Ejecutar y monitorear:

In [ ]:
# Iniciar scheduler (en terminal separado)
!airflow scheduler

# Iniciar webserver
!airflow webserver --port 8080

# Ejecutar DAG manualmente
!airflow dags unpause saludo_diario
!airflow dags trigger saludo_diario

Ver resultados:

In [ ]:
# Ver logs de ejecución
# Visitar http://localhost:8080 en navegador
# Ir a DAGs → saludo_diario → Graph View para ver flujo
# Ir a Tree View para ver historial de ejecuciones

#### Ejemplo: En Google Colab

In [ ]:
# 1. Instalar Airflow y dependencias (esto tarda un par de minutos)
!pip install apache-airflow==2.10.0 --quiet

# 2. Configurar la variable de entorno para la carpeta de Airflow
import os
os.environ['AIRFLOW_HOME'] = '/content/airflow'

# 3. Inicializar la base de datos
!airflow db init

# 4. Crear el usuario administrador
!airflow users create \
  --username admin \
  --firstname Admin \
  --lastname User \
  --role Admin \
  --email admin@example.com \
  --password admin

In [7]:
# Crear carpeta de dags
!mkdir -p /content/airflow/dags

# Escribir el archivo del DAG
with open('/content/airflow/dags/mi_primer_dag.py', 'w') as f:
    f.write("""
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

def saludar():
    print("¡Hola desde Airflow!")
    return "Saludo completado"

with DAG(
    'saludo_diario',
    description='DAG que saluda cada día',
    schedule_interval='@daily',
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=['ejemplo', 'saludo']
) as dag:

    tarea_bash = BashOperator(
        task_id='tarea_bash',
        bash_command='echo "Ejecutando tarea bash"'
    )

    tarea_python = PythonOperator(
        task_id='tarea_python',
        python_callable=saludar
    )

    tarea_esperar = BashOperator(
        task_id='tarea_esperar',
        bash_command='sleep 5'
    )

    tarea_bash >> tarea_python >> tarea_esperar
""")

In [10]:
# 1. Instalar localtunnel para poder entrar a la web desde afuera
!npm install -g localtunnel --quiet

# 2. Iniciar el Scheduler en segundo plano
import subprocess
subprocess.Popen(['airflow', 'scheduler'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 3. Iniciar el Webserver en segundo plano (puerto 8080)
subprocess.Popen(['airflow', 'webserver', '--port', '8080'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 4. Obtener tu IP pública (la necesitarás para entrar al túnel)
print("Tu IP para el túnel es:")
!curl ipv4.icanhazip.com

# 5. Crear el túnel para acceder a Airflow
!lt --port 8080

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 946ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Tu IP para el túnel es:
35.229.97.231
your url is: https://cold-parrots-raise.loca.lt
^C


In [ ]:
!airflow dags unpause saludo_diario
!airflow dags trigger saludo_diario
# Espera 10 segundos y mira el estado
!airflow tasks state saludo_diario tarea_python 2024-01-01

In [ ]:
!curl https://loca.lt/mytunnelpassword

### DAGs y Dependencias

**Estructura de un DAG Complejo**

Elementos de un DAG bien diseñado:

- Configuración clara: Nombre, descripción, schedule
- Tareas modulares: Cada tarea hace una cosa específica
- Dependencias explícitas: Flujo claro de ejecución
- Manejo de errores: Reintentos y alertas configurados

Ejemplo de DAG estructurado:

```
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_team',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'email_on_failure': True,
    'email_on_retry': False,
    'retries': 2,
    'retry_delay': timedelta(minutes=5)
}

dag = DAG(
    'etl_pipeline_diario',
    default_args=default_args,
    description='Pipeline ETL completo diario',
    schedule_interval='@daily',
    catchup=False,
    max_active_runs=1,
    tags=['etl', 'diario', 'produccion']
)
```



**Tipos de Dependencias**

Formas de definir dependencias:

1. Sintaxis de flechas:
```
# Dependencias simples
tarea_a >> tarea_b  # A luego B
tarea_c << tarea_d  # D luego C (equivalente)

# Dependencias múltiples
tarea_a >> [tarea_b, tarea_c]  # A luego B y C en paralelo
[tarea_x, tarea_y] >> tarea_z  # X e Y luego Z
```
2. Método set_upstream/downstream:
```
# Más explícito para casos complejos
tarea_b.set_upstream(tarea_a)    # A antes que B
tarea_a.set_downstream(tarea_c)  # A antes que C
```
Patrones comunes:

- Secuencial: A → B → C
- Paralelo: A → [B, C] → D
- Diamond: A → [B, C] → D
- Fan-out/fan-in: Uno → muchos → uno


**Operadores Principales**

Tipos de operadores en Airflow:

1. PythonOperator: Ejecuta funciones Python
```
def procesar_datos(**context):
    print("Procesando datos...")
    return "Completado"

tarea_python = PythonOperator(
    task_id='procesar_datos',
    python_callable=procesar_datos,
    provide_context=True,
    dag=dag
)
```
2. BashOperator: Ejecuta comandos shell
```
tarea_bash = BashOperator(
    task_id='limpiar_archivos',
    bash_command='rm -f /tmp/*.tmp',
    dag=dag
)
```
3. DummyOperator: Para estructura del grafo
```
from airflow.operators.dummy import DummyOperator

inicio = DummyOperator(task_id='inicio', dag=dag)
fin = DummyOperator(task_id='fin', dag=dag)
```

#### Ejemplo: Construir DAG con dependencias complejas

DAG de procesamiento de ventas:

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

def extraer_ventas():
    """Simular extracción de datos de ventas"""
    print("Extrayendo datos de ventas...")
    return {"registros": 1000}

def validar_datos(ventas):
    """Validar calidad de datos"""
    print(f"Validando {ventas['registros']} registros...")
    return {"validos": 950, "errores": 50}

def transformar_datos(datos):
    """Aplicar transformaciones de negocio"""
    print(f"Transformando {datos['validos']} registros válidos...")
    return {"transformados": datos['validos']}

def cargar_data_warehouse(transformados):
    """Cargar a data warehouse"""
    print(f"Cargando {transformados['transformados']} registros...")
    return {"cargados": transformados['transformados']}

def enviar_reporte(resultado):
    """Enviar reporte de ejecución"""
    print(f"Pipeline completado: {resultado['cargados']} registros procesados")

# Configurar DAG
dag = DAG(
    'pipeline_ventas_complejo',
    description='Pipeline ETL de ventas con dependencias complejas',
    schedule_interval='@daily',
    start_date=datetime(2024, 1, 1),
    catchup=False,
    default_args={
        'retries': 2,
        'retry_delay': timedelta(minutes=5)
    }
)

# Tareas de extracción (pueden ejecutarse en paralelo)
extraer_api = PythonOperator(
    task_id='extraer_api_ventas',
    python_callable=extraer_ventas,
    dag=dag
)

extraer_db = PythonOperator(
    task_id='extraer_db_productos',
    python_callable=lambda: {"productos": 500},
    dag=dag
)

# Tarea de preparación
preparar_entorno = BashOperator(
    task_id='preparar_entorno',
    bash_command='mkdir -p /tmp/etl_ventas',
    dag=dag
)

# Tareas de validación (dependen de extracción)
validar_api = PythonOperator(
    task_id='validar_datos_api',
    python_callable=lambda: validar_datos({"registros": 1000}),
    dag=dag
)

validar_db = PythonOperator(
    task_id='validar_datos_db',
    python_callable=lambda: {"productos_validos": 480},
    dag=dag
)

# Tareas de transformación (dependen de validación)
transformar_ventas = PythonOperator(
    task_id='transformar_ventas',
    python_callable=lambda: transformar_datos({"validos": 950}),
    dag=dag
)

transformar_productos = PythonOperator(
    task_id='transformar_productos',
    python_callable=lambda: {"productos_transformados": 480},
    dag=dag
)

# Tarea de join (une ventas y productos)
join_datos = PythonOperator(
    task_id='join_ventas_productos',
    python_callable=lambda: {"registros_completos": 920},
    dag=dag
)

# Carga final
cargar_dw = PythonOperator(
    task_id='cargar_data_warehouse',
    python_callable=lambda: cargar_data_warehouse({"transformados": 920}),
    dag=dag
)

# Reporte final
enviar_reporte = PythonOperator(
    task_id='enviar_reporte_ejecucion',
    python_callable=lambda: enviar_reporte({"cargados": 920}),
    dag=dag
)

# Definir dependencias complejas
# Preparación inicial
preparar_entorno >> [extraer_api, extraer_db]

# Extracción → Validación
extraer_api >> validar_api
extraer_db >> validar_db

# Validación → Transformación
validar_api >> transformar_ventas
validar_db >> transformar_productos

# Transformaciones → Join
[transformar_ventas, transformar_productos] >> join_datos

# Join → Carga → Reporte
join_datos >> cargar_dw >> enviar_reporte

Visualizar el grafo de dependencias:

In [ ]:
# Ver el DAG en Airflow Web UI
# Ir a Graph View para ver el flujo visual

# El grafo debería verse así:
# preparar_entorno → [extraer_api, extraer_db]
# extraer_api → validar_api → transformar_ventas ↘
# extraer_db → validar_db → transformar_productos ↘ → join_datos → cargar_dw → enviar_reporte

Probar diferentes escenarios:

In [ ]:
# Para probar: airflow dags test pipeline_ventas_complejo
# Para ejecutar: airflow dags trigger pipeline_ventas_complejo
# Para ver logs: airflow tasks logs pipeline_ventas_complejo enviar_reporte_ejecucion 2024-01-01

### Operadores y Sensores

**Operadores Comunes**



Operadores más utilizados en Airflow:

1. BashOperator - Ejecuta comandos shell
```
from airflow.operators.bash import BashOperator

tarea_bash = BashOperator(
    task_id='procesar_archivos',
    bash_command='find /data -name "*.csv" -exec wc -l {} \;',
    dag=dag
)
```
2. PythonOperator - Ejecuta funciones Python
```
from airflow.operators.python import PythonOperator

def mi_funcion(execution_date, **context):
    print(f"Ejecutando en: {execution_date}")
    return "Completado"

tarea_python = PythonOperator(
    task_id='ejecutar_logica',
    python_callable=mi_funcion,
    provide_context=True,
    dag=dag
)
```
3. EmailOperator - Envía correos electrónicos
```
from airflow.operators.email import EmailOperator

tarea_email = EmailOperator(
    task_id='notificar_equipo',
    to='data@empresa.com',
    subject='Pipeline completado',
    html_content='<p>El pipeline ETL ha finalizado exitosamente.</p>',
    dag=dag
)
```

**Sensores para Esperar Condiciones**


¿Qué son los sensores?

Los sensores esperan hasta que se cumpla una condición antes de continuar.

Ejemplos comunes:

1. FileSensor - Espera archivo
    ```
    from airflow.sensors.filesystem import FileSensor

    esperar_archivo = FileSensor(
        task_id='esperar_datos_entrada',
        filepath='/data/input/datos.csv',
        poke_interval=30,  # Revisar cada 30 segundos
        timeout=3600,      # Máximo 1 hora
        dag=dag
    )
    ```
2. HttpSensor - Espera respuesta HTTP
    ```
    from airflow.sensors.http_sensor import HttpSensor

    esperar_api = HttpSensor(
        task_id='esperar_api_disponible',
        http_conn_id='mi_api',
        endpoint='health',
        poke_interval=60,
        timeout=300,
        dag=dag
    )
    ```

**Operadores Personalizados**

¿Por qué crear operadores personalizados?

Para encapsular lógica reutilizable y mantener código limpio.

Estructura básica:


```
from airflow.models.baseoperator import BaseOperator
from airflow.utils.decorators import apply_defaults

class MiOperadorPersonalizado(BaseOperator):
    
    @apply_defaults
    def __init__(self, mi_parametro, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mi_parametro = mi_parametro
    
    def execute(self, context):
        self.log.info(f"Ejecutando con parámetro: {self.mi_parametro}")
        
        # Lógica del operador
        resultado = f"Procesado con {self.mi_parametro}"
        
        return resultado
```



#### Ejemplo: Decorador

In [32]:
import time

def medir_tiempo(funcion):
    def envoltura(*args, **kwargs):
        inicio = time.time()
        resultado = funcion(*args, **kwargs) # Ejecuta la función original
        fin = time.time()
        print(f"La función tardó {fin - inicio} segundos")
        return resultado
    return envoltura

@medir_tiempo
def proceso_pesado():
    time.sleep(2)
    print("Proceso terminado")

proceso_pesado()

Proceso terminado
La función tardó 2.00850248336792 segundos


#### Ejemplo: Crear DAG con operadores y sensores

DAG de procesamiento con sensores:

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor
from datetime import datetime, timedelta

def procesar_datos():
    print("Procesando datos de ventas...")
    return "Datos procesados"

def generar_reporte():
    print("Generando reporte ejecutivo...")
    return "Reporte generado"

dag = DAG(
    'pipeline_con_sensores',
    description='Pipeline que espera archivos antes de procesar',
    schedule_interval='@hourly',
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# Sensor que espera archivo de entrada
esperar_datos = FileSensor(
    task_id='esperar_archivo_datos',
    filepath='/tmp/datos_ventas.csv',
    poke_interval=60,    # Revisar cada minuto
    timeout=3600,        # Máximo 1 hora
    mode='poke',         # Modo de verificación
    dag=dag
)

# Procesar datos una vez que el archivo llegue
procesar = PythonOperator(
    task_id='procesar_datos_ventas',
    python_callable=procesar_datos,
    dag=dag
)

# Generar reporte
reporte = PythonOperator(
    task_id='generar_reporte',
    python_callable=generar_reporte,
    dag=dag
)

# Limpiar archivos temporales
limpiar = BashOperator(
    task_id='limpiar_archivos',
    bash_command='rm -f /tmp/datos_ventas.csv',
    dag=dag
)

# Definir flujo: esperar → procesar → reportar → limpiar
esperar_datos >> procesar >> reporte >> limpiar

Crear operador personalizado:

In [ ]:
from airflow.models.baseoperator import BaseOperator
from airflow.utils.decorators import apply_defaults
import pandas as pd

class ValidadorDatosOperator(BaseOperator):

    @apply_defaults
    def __init__(self, archivo_entrada, umbral_calidad=0.9, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.archivo_entrada = archivo_entrada
        self.umbral_calidad = umbral_calidad

    def execute(self, context):
        self.log.info(f"Validando archivo: {self.archivo_entrada}")

        # Leer datos
        try:
            df = pd.read_csv(self.archivo_entrada)
        except Exception as e:
            raise Exception(f"Error leyendo archivo: {e}")

        # Validaciones
        total_registros = len(df)
        registros_completos = df.dropna().shape[0]
        calidad = registros_completos / total_registros

        self.log.info(f"Calidad de datos: {calidad:.2%}")

        if calidad < self.umbral_calidad:
            raise Exception(f"Calidad insuficiente: {calidad:.2%} < {self.umbral_calidad:.2%}")

        return {
            'registros_totales': total_registros,
            'registros_validos': registros_completos,
            'calidad': calidad
        }

# Usar operador personalizado en DAG
validar_datos = ValidadorDatosOperator(
    task_id='validar_datos_ventas',
    archivo_entrada='/tmp/datos_ventas.csv',
    umbral_calidad=0.95,
    dag=dag
)

# Actualizar dependencias
esperar_datos >> validar_datos >> procesar >> reporte >> limpiar

Probar el DAG:

In [ ]:
# Crear archivo de prueba
echo "id,nombre,ventas
1,Producto A,100
2,Producto B,200
3,Producto C,150" > /tmp/datos_ventas.csv

# Ejecutar DAG
airflow dags trigger pipeline_con_sensores

# Monitorear en web UI

### Monitoreo y Alertas

### Pipelines Complejos y Best Practices

## Week 3 - Proyecto Final - Pipeline Completo

### Diseño de Arquitectura Completa

### Implementación de Pipeline End-to-End

### Validación y Testing

### Optimización y Performance

### Documentación y Presentación

## Week 4 - Carrea Profesional y Próximos Pasos

### Estrategias de Despliegue (CI/CD)

### Monitoreo y Observabilidad

### Gestión de Incidentes y Recuperación

### Actualización y Escalabilidad de Pipelines

### Documentación y Presentación